# Aula 22 · Lab bônus: Lakeflow Connect (resolução comentada)

## 1. O pipeline de ingestão
1. **Jobs & Pipelines → Create → ETL pipeline**.
2. **Name:** `growdev_ingestao_gerenciada`.
3. **Default catalog:** `growdev` · **Default schema:** `bronze`.
4. **Source code:** **Add existing assets** → selecione `Projeto/pipeline_ingestao/ingestao_loans.sql`.
5. **Settings → Configuration → Add configuration:** `catalogo` = `growdev`.
6. **Run pipeline**.

O coração do pipeline é o `read_files` em modo `STREAM`:

```sql
CREATE OR REFRESH STREAMING TABLE loans_ingestao_gerenciada
AS SELECT *, current_timestamp() AS _ingested_at, _metadata.file_path AS _source_file
FROM STREAM read_files('/Volumes/${catalogo}/bronze/landing/loans/', format => 'parquet');
```

`STREAMING TABLE` + `STREAM` faz o pipeline **guardar o que já leu** (checkpoint). Na próxima execução ele só busca arquivos novos, sem a lógica manual de `_source_file` que escrevemos na bronze.

### E com um conector gerenciado (MySQL, SQL Server, Salesforce...)?
**Jobs & Pipelines → Create → Ingestion pipeline** → escolha o conector → informe a **conexão** (credenciais guardadas no Unity Catalog, nunca no código) → selecione as tabelas → destino. O Databricks gera o pipeline, e no job ele entra **exatamente igual**: uma tarefa do tipo **Pipeline**.

## 2. Primeira etapa do orquestrador
No `growdev_orquestrador`:
1. **+ Add task → Pipeline** → `ingestao_gerenciada` → **Pipeline:** `growdev_ingestao_gerenciada`.
2. Edite `executar_ingestao` → **Depends on:** `ingestao_gerenciada`.

```
growdev_orquestrador
   ingestao_gerenciada (Pipeline) → executar_ingestao (Run Job) → executar_transformacao (Run Job)
```

## 3 e 4. Executando e conferindo a ingestão incremental

In [ ]:
%sql
SELECT _source_file, COUNT(*) AS linhas, MIN(_ingested_at) AS ingerido_em
FROM growdev.bronze.loans_ingestao_gerenciada
GROUP BY _source_file
ORDER BY ingerido_em

Depois de entregar um lote novo e rodar o orquestrador de novo, aparece **uma linha a mais** nesta consulta (o arquivo novo), e os arquivos antigos mantêm o mesmo horário de ingestão.

## Como código
Veja `resources/growdev_ingestao_gerenciada.pipeline.yml` (o pipeline) e `resources/growdev_orquestrador.job.yml` (a tarefa `pipeline_task` no início do orquestrador).